# Shift Factors

Companion notebook for Tutorial 08. Compute and interpret the sensitivity of
branch, interface, and flowgate flows to a 1 MW injection at every bus in
the network.

Shift factors (PTDFs) underpin congestion pricing, flowgate monitoring,
generator interconnection studies, and transfer capability analysis.

## Single-Branch Shift Factors

Start with the 9-bus case for intuition. Compute the PTDF for one branch
across all buses.

In [ ]:
import surge
from surge.dc import BranchKey, PtdfRequest

net = surge.case9()

# Pick a branch to monitor: bus 4 -> bus 5, circuit "1"
branch = BranchKey(4, 5)

ptdf = surge.dc.compute_ptdf(net, PtdfRequest(
    monitored_branches=(branch,),
))

print("PTDF shape:", ptdf.ptdf.shape)
print("Bus numbers:", ptdf.bus_numbers)
print()
for bus, sf in zip(ptdf.bus_numbers, ptdf.ptdf[0]):
    print(f"  Bus {bus:3d}: {sf:+.4f}")

The shift factor at the slack bus is zero by definition (single-slack
semantics). Buses electrically close to the monitored branch have larger
absolute values.

## Verify Against DC Power Flow

Perturb injection at one bus and confirm the flow change matches the
shift factor prediction.

In [ ]:
# Baseline DC power flow
dc_base = surge.solve_dc_pf(net, surge.DcPfOptions())

# Perturb: add 10 MW injection at bus 5 (reduce load by 10 MW)
perturbed = surge.case9()
original_load = perturbed.bus_pd[perturbed.bus_numbers.index(5)]
perturbed.set_bus_load(5, original_load - 10.0)
dc_pert = surge.solve_dc_pf(perturbed, surge.DcPfOptions())

branch_idx = net.branch_index(4, 5, "1")
delta_flow = dc_pert.branch_p_mw[branch_idx] - dc_base.branch_p_mw[branch_idx]
bus_col = ptdf.bus_numbers.index(5)
predicted = ptdf.ptdf[0, bus_col] * 10.0

print(f"Actual flow change:    {delta_flow:+.4f} MW")
print(f"PTDF prediction:       {predicted:+.4f} MW")

## Interface Shift Factors

An interface is a weighted combination of branches that defines a
transmission boundary. Its shift factor is the matching weighted sum of
branch PTDFs.

In [ ]:
import numpy as np

net30 = surge.case30()

# Interface: 60% of branch 27->30 + 40% of branch 29->30
iface_branches = (BranchKey(27, 30), BranchKey(29, 30))
iface_weights = np.array([0.6, 0.4])

ptdf30 = surge.dc.compute_ptdf(net30, PtdfRequest(
    monitored_branches=iface_branches,
))

# Interface shift factor = weighted sum of branch PTDF rows
iface_sf = iface_weights @ ptdf30.ptdf

print("Top 5 buses by absolute interface shift factor:")
order = np.argsort(-np.abs(iface_sf))
for col in order[:5]:
    print(f"  Bus {ptdf30.bus_numbers[col]:3d}: {iface_sf[col]:+.4f}")

## Distributed Slack

Single-slack shift factors assume all mismatch is absorbed at the reference
bus. Compare against weighted and headroom-based slack policies.

In [ ]:
from surge.dc import SlackPolicy

branch_68 = BranchKey(6, 8)

sf_single = surge.dc.compute_ptdf(net30, PtdfRequest(
    monitored_branches=(branch_68,),
    slack=SlackPolicy.single(),
))

sf_dist = surge.dc.compute_ptdf(net30, PtdfRequest(
    monitored_branches=(branch_68,),
    slack=SlackPolicy.weights({1: 0.5, 2: 0.3, 13: 0.2}),
))

sf_headroom = surge.dc.compute_ptdf(net30, PtdfRequest(
    monitored_branches=(branch_68,),
    slack=SlackPolicy.headroom(),
))

print(f"{'Bus':>5s} {'Single':>10s} {'Weighted':>10s} {'Headroom':>10s}")
for i, bus in enumerate(sf_single.bus_numbers):
    s = sf_single.ptdf[0, i]
    w = sf_dist.ptdf[0, i]
    h = sf_headroom.ptdf[0, i]
    if abs(s) > 0.01 or abs(w) > 0.01 or abs(h) > 0.01:
        print(f"{bus:5d} {s:+10.4f} {w:+10.4f} {h:+10.4f}")

Under distributed slack, the slack-bus column is no longer zero—every bus
gets a nonzero shift factor because the balancing response is spread out.

## Flowgate Shift Factors

A flowgate is a **monitored element paired with a specific contingency**.
What makes it a flowgate—rather than just a branch or interface—is the
contingency. Its shift factor is therefore an OTDF, not a base-case PTDF.

In [ ]:
from surge.dc import OtdfRequest

# Flowgate: monitor branch 27->30 for the loss of branch 29->30
monitored = BranchKey(27, 30)
contingency = BranchKey(29, 30)

# Base-case shift factor (branch only, no contingency)
ptdf_pre = surge.dc.compute_ptdf(net30, PtdfRequest(
    monitored_branches=(monitored,),
))

# Flowgate shift factor = OTDF (monitored element + contingency)
otdf = surge.dc.compute_otdf(net30, OtdfRequest(
    monitored_branches=(monitored,),
    outage_branches=(contingency,),
))

pre = ptdf_pre.ptdf[0]
post = otdf.otdf[0, 0]

print("Buses with largest shift factor increase under flowgate contingency:")
delta = np.abs(post) - np.abs(pre)
order = np.argsort(-delta)
for col in order[:5]:
    bus = ptdf_pre.bus_numbers[col]
    print(f"  Bus {bus:3d}: {pre[col]:+.4f} -> {post[col]:+.4f}  (delta = {delta[col]:+.4f})")

When the contingency trips branch 29→30, flow that previously split between
the two parallel paths now concentrates on 27→30. The OTDF captures this
redistribution. Flowgate monitoring works the same way: precompute
OTDFs for each credible contingency and check whether post-contingency flow
would violate the flowgate limit.

## Prepared Study

When computing multiple sensitivity products, factor the network once and
reuse. The `to_dataframe()` method is convenient for tabular analysis.

In [ ]:
net118 = surge.case118()
study = surge.dc.prepare_study(net118)

ptdf_full = study.compute_ptdf()
lodf_full = study.compute_lodf()
otdf_118 = study.compute_otdf(OtdfRequest(
    monitored_branches=(BranchKey(69, 75), BranchKey(69, 77)),
    outage_branches=(BranchKey(65, 68),),
))

print("PTDF:", ptdf_full.ptdf.shape)
print("LODF:", lodf_full.lodf.shape)
print("OTDF:", otdf_118.otdf.shape)
print()

df = ptdf_full.to_dataframe()
print(df.head())